# Pitcher Handedness From FanGraphs

FanGraphs pitching data includes a `Throws` column when the local scraper is called with `advanced=True`. This notebook shows how to load that data and look up a pitcher's handedness by name.

## Imports

The path helper lets this notebook work whether it is launched from the repository root or from the `examples` folder.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

repo_root = Path.cwd()
if (repo_root / "src").exists():
    sys.path.insert(0, str(repo_root / "src"))
elif (repo_root.parent / "src").exists():
    sys.path.insert(0, str(repo_root.parent / "src"))

import mlbstatscraping as ss

pd.set_option("display.max_columns", 100)

## Load Pitching Data

Use `advanced=True` so the FanGraphs response is kept as a full player table, including `Throws`.

In [ ]:
YEAR = "2026"

pitching = ss.get_fangraphs_data("pitching", [YEAR], advanced=True)
pitching.shape

Confirm that the key identity columns are present.

In [ ]:
identity_columns = ["Name", "Team", "Throws", "playerid", "xMLBAMID", "G", "IP"]
pitching.loc[:, [col for col in identity_columns if col in pitching.columns]].head(10)

## Look Up a Pitcher

This helper first tries an exact case-insensitive name match. If that fails, it returns partial name matches so you can see whether the player is listed under a slightly different name.

In [ ]:
def get_pitcher_handedness(name, df=pitching):
    lookup = name.strip().casefold()
    names = df["Name"].astype(str).str.strip()

    exact = df.loc[names.str.casefold() == lookup].copy()
    if not exact.empty:
        return exact.loc[:, ["Name", "Team", "Throws", "Year"]].drop_duplicates()

    partial = df.loc[names.str.casefold().str.contains(lookup, regex=False, na=False)].copy()
    if not partial.empty:
        return partial.loc[:, ["Name", "Team", "Throws", "Year"]].drop_duplicates()

    return pd.DataFrame(columns=["Name", "Team", "Throws", "Year"])


get_pitcher_handedness("Tarik Skubal")

For repeated lookups, build a small handedness table once and query it locally.

In [ ]:
pitcher_handedness = (
    pitching.loc[:, ["Name", "Team", "Throws", "Year"]]
    .dropna(subset=["Name", "Throws"])
    .drop_duplicates()
    .sort_values(["Name", "Team"])
    .reset_index(drop=True)
)

pitcher_handedness.head(20)

In [ ]:
names_to_check = ["Tarik Skubal", "Chris Sale", "Logan Webb"]

pd.concat(
    [get_pitcher_handedness(name, pitcher_handedness) for name in names_to_check],
    ignore_index=True,
)